# 02. Conformal Calibration: Uncertainty Quantification & XAI
## Ваша зона (Тимлид: MAPIE)

**Цель этапа:**
1. Загрузка логитов и вероятностей из `outputs/predictions.csv` (или генерация калибровочного разбиения).
2. Конформная калибровка с гарантией покрытия $1 - \alpha$ (Split Conformal, LAC, APS, MAPIE).
3. Построение множеств предсказаний $C(x)$.
4. Оценка метрик: эмпирическое покрытие (Coverage) и средний размер множеств $|C(x)|$ (Efficiency).
5. Связка с объяснимым ИИ (Grad-CAM): анализ пространственного согласования/разногласия при неоднозначных множествах ($|C(x)| > 1$).

In [ ]:
# 1. Импорт библиотек и модулей проекта
import sys
from pathlib import Path

# Добавляем корень проекта в sys.path для импорта src
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# MAPIE
try:
    from mapie.classification import MapieClassifier
    print("MAPIE успешно импортирован!")
except ImportError:
    print("MAPIE не установлен. Будет использована встроенная реализация src.conformal.")

# Встроенные модули проекта
from src.conformal import compute_nonconformity_scores, calibrate_threshold, predict_conformal_sets
from src.metrics import conformal_classification_metrics, cam_conformal_alignment_score

sns.set_theme(style="whitegrid")

### 2. Загрузка данных / предсказаний baseline модели

In [ ]:
OUTPUTS_DIR = Path("../outputs")
pred_path = OUTPUTS_DIR / "predictions.csv"

CLASS_NAMES = ["Normal", "Atelectasis", "Cardiomegaly", "Effusion", "Pneumonia"]
num_classes = len(CLASS_NAMES)

if pred_path.exists():
    df_preds = pd.read_csv(pred_path)
    prob_cols = [c for c in df_preds.columns if c.startswith("prob_")]
    probs = df_preds[prob_cols].values
    targets = df_preds["true_label"].values
    print(f"Загружено {len(probs)} предсказаний из {pred_path}")
else:
    print("Файл predictions.csv не найден. Генерируем синтетический срез вероятностей для демонстрации калибровки.")
    np.random.seed(42)
    n_samples = 1000
    logits = np.random.randn(n_samples, num_classes) * 2.0
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
    targets = np.random.choice(num_classes, size=n_samples, p=[0.4, 0.15, 0.15, 0.15, 0.15])

### 3. Разделение на калибровочную (Calibration) и тестовую (Test) выборки

In [ ]:
probs_cal, probs_test, y_cal, y_test = train_test_split(
    probs, targets, test_size=0.5, random_state=42, stratify=targets
)
print(f"Калибровочная выборка: {len(y_cal)} образцов")
print(f"Тестовая выборка: {len(y_test)} образцов")

### 4. Конформная калибровка (Split Conformal / LAC)

In [ ]:
alpha = 0.10  # 90% целевой уровень доверия

# 1. Вычисление non-conformity scores на калибровочной выборке
scores_cal = compute_nonconformity_scores(probs_cal, y_cal, method="lac")

# 2. Вычисление квантиля q_hat
q_hat = calibrate_threshold(scores_cal, alpha=alpha)
print(f"Уровень ошибки alpha = {alpha} (Желаемое покрытие: {1 - alpha:.1%})")
print(f"Калиброванный квантиль q_hat: {q_hat:.4f}")
print(f"Пороговое значение вероятности (1 - q_hat): {1.0 - q_hat:.4f}")

### 5. Построение конформных множеств C(x) и расчет метрик

In [ ]:
test_sets = predict_conformal_sets(probs_test, q_hat=q_hat, method="lac", class_names=CLASS_NAMES)
y_test_names = [CLASS_NAMES[idx] for idx in y_test]

metrics = conformal_classification_metrics(test_sets, y_test_names, alpha=alpha)
print("=== Результаты конформной калибровки ===")
print(f"Целевое покрытие:      {metrics['target_coverage']:.1%}")
print(f"Эмпирическое покрытие: {metrics['empirical_coverage']:.1%}")
print(f"Разрыв покрытия (gap): {metrics['coverage_gap']:+.4f}")
print(f"Средний размер C(x):   {metrics['avg_set_size']:.2f}")
print(f"Распределение размеров: {metrics['size_distribution']}")

### 6. Визуализация распределения размеров множеств |C(x)|

In [ ]:
sizes = [len(s) for s in test_sets]

plt.figure(figsize=(8, 4))
sns.countplot(x=sizes, palette="Blues_d")
plt.title(f"Распределение размера предсказательного множества |C(x)| (alpha={alpha})")
plt.xlabel("Размер множества")
plt.ylabel("Количество наблюдений")
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "conformal_set_sizes.png", dpi=300)
plt.show()

### 7. Связка с объяснимым ИИ (Grad-CAM XAI alignment)

In [ ]:
# Демонстрация связки: для неопределенных прогнозов (|C(x)| > 1)
# вычисляем степень наложения тепловых карт Grad-CAM разных диагнозов
ambiguous_indices = [i for i, s in enumerate(test_sets) if len(s) > 1]
print(f"Количество неопределенных предсказаний (|C(x)| > 1): {len(ambiguous_indices)}")

if ambiguous_indices:
    sample_idx = ambiguous_indices[0]
    sample_set = test_sets[sample_idx]
    print(f"Пример множества диагнозов для образца #{sample_idx}: {sample_set}")
    
    # Имитация Grad-CAM карт для демонстрации расчета метрики
    mock_heatmaps = {
        c: np.random.rand(224, 224)
        for c in sample_set
    }
    alignment = cam_conformal_alignment_score(mock_heatmaps, sample_set, method="iou")
    print("Grad-CAM метрика согласованности:", alignment)